# Rebuild QQQ's Historical Stock Universe Without Survivorship Bias

## Goal

The goal is to solve the **survivorship bias problem** in historical QQQ analysis.

An **index constituent** is a company or security included in an index on a specific date. A backtest is biased when it applies today's constituents to earlier years, because that excludes stocks that were later removed, acquired, delisted, or renamed.

QQQ tracks the Nasdaq-100, so this notebook reconstructs the Nasdaq-100 constituent list at each verified change date. This creates a point-in-time research universe for QQQ backtests; it is not a daily ETF holdings ledger.

**Verified coverage:** March 20, 2015 through July 7, 2026.

## The Problem: A Current-Only Backtest Looks Into the Future

The constituent lists changed substantially:

| Comparison | Tickers | Percentage |
|---|---:|---:|
| In the March 20, 2015 list | 107 | 100.0% |
| Same ticker symbol in both the 2015 and latest lists | 43 | 40.2% of the 2015 list |
| In 2015 but absent from the latest list | 64 | **59.8% of the 2015 list** |
| In the latest list but absent from the 2015 list | 60 | **58.3% of the latest 103 tickers** |

A backtest that uses only the latest constituents creates two upward biases:

1. **Pre-inclusion look-ahead bias:** if a latest constituent has price history before its actual ADD date, the backtest can buy it before it entered the Nasdaq-100. This gives the strategy early access to companies that later became successful enough to enter the index.
2. **Missing-exit survivorship bias:** the backtest excludes 2015 constituents that were later removed, acquired, delisted, or replaced. It therefore avoids losses and drawdowns that the historical portfolio could actually have experienced.

Together, these errors can materially overstate historical returns and understate risk.

> **Ticker-level note:** 'absent from the latest list' does not mean every company was delisted. The 64 tickers include index removals, acquisitions, ticker changes, and successor securities. File 04 explains those cases.

## Workflow

We will complete the following steps in order:

| Step | Input | Output | Purpose |
|---:|---|---|---|
| 0 | Starting constituent list | `00_starting_constituents_2015-03-20.csv` | Define the first known index snapshot |
| 1 | Official PDFs | PDF inventory and text | Review the source documents |
| 2 | PDF text + GPT-5.5 | `02_constituent_changes.xlsx` | Extract ticker additions and removals |
| 3 | Starting list + changes | `03_historical_constituent_matrix_1_0.xlsx` | Build the complete historical 1/0 matrix |
| 4 | Removal rows | `04_index_exits_and_ticker_changes.csv` | Explain index exits and ticker changes |
| 5 | Last matrix column | `05_current_constituents_only.xlsx` | Create a current-constituents-only view |

> **Use file 03 for survivorship-bias-aware backtests.** File 05 intentionally keeps only today's constituents and must not be used as the historical universe.

## Setup

Required packages: `openai`, `pydantic`, `pypdf`, `pandas`, and `openpyxl`.

The API key is never stored in this notebook. It is read from `OPENAI_API_KEY` or entered securely at runtime.

The default setting reuses the validated change workbook, so the notebook can run without API cost. Set `RUN_GPT_EXTRACTION = True` when you want to rebuild file 02 from the PDFs.

Keep `01_PDF_sources` in the same folder as this notebook. The complete `final` folder is portable and does not use an absolute path.

In [1]:
from __future__ import annotations

import getpass
import json
import os
import re
from pathlib import Path
from typing import Literal

import pandas as pd
from IPython.display import display
from openai import OpenAI
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from pydantic import BaseModel
from pypdf import PdfReader

MODEL_NAME = 'gpt-5.5'
RUN_GPT_EXTRACTION = False  # Change to True to rebuild file 02 with the API.
MAX_PDFS = None             # Use 3 for a small API test.
MAX_CHARS_PER_PDF = 120_000

# These values describe the validated reference result.
EXPECTED_STARTING_TICKERS = 107
EXPECTED_CHANGE_ROWS = 262
EXPECTED_HISTORICAL_TICKERS = 223
EXPECTED_DATE_COLUMNS = 74
EXPECTED_CURRENT_CONSTITUENTS = 103
EXPECTED_SHARED_TICKERS = 43
EXPECTED_START_ONLY_TICKERS = 64
EXPECTED_CURRENT_ONLY_TICKERS = 60

### Locate the input and output folders

In [2]:
def find_final_folder(start: Path | None = None) -> Path:
    """Find the folder that contains the starting constituent list."""
    start = (start or Path.cwd()).resolve()
    candidate_folders = [start, start / 'daehyeon', start / 'final', start / 'final' / 'daehyeon', *start.parents]
    for folder in candidate_folders:
        if (folder / '00_starting_constituents_2015-03-20.csv').is_file():
            return folder
    raise FileNotFoundError('Could not find the final folder.')

FINAL_DIR = find_final_folder()

# The source PDFs travel with this notebook inside the portable final folder.
PDF_DIR = FINAL_DIR / '01_PDF_sources'
if not PDF_DIR.is_dir():
    raise FileNotFoundError('Expected 01_PDF_sources inside the final folder.')

STARTING_PATH = FINAL_DIR / '00_starting_constituents_2015-03-20.csv'
CHANGES_PATH = FINAL_DIR / '02_constituent_changes.xlsx'
MATRIX_PATH = FINAL_DIR / '03_historical_constituent_matrix_1_0.xlsx'
EXITS_PATH = FINAL_DIR / '04_index_exits_and_ticker_changes.csv'
CURRENT_PATH = FINAL_DIR / '05_current_constituents_only.xlsx'
CACHE_DIR = FINAL_DIR / '_gpt_cache'

print('Working folder:', FINAL_DIR.name)
print('PDF folder:    ', PDF_DIR.relative_to(FINAL_DIR))

Working folder: daehyeon
PDF folder:     01_PDF_sources


In [3]:
def format_excel(path: Path, matrix_sheet: str | None = None) -> None:
    """Add a readable header, filters, and frozen rows."""
    workbook = load_workbook(path)
    for sheet in workbook.worksheets:
        sheet.freeze_panes = 'B2' if sheet.title == matrix_sheet else 'A2'
        sheet.auto_filter.ref = sheet.dimensions
        sheet.sheet_view.showGridLines = False

        for cell in sheet[1]:
            cell.fill = PatternFill('solid', fgColor='087F73')
            cell.font = Font(color='FFFFFF', bold=True)
            cell.alignment = Alignment(horizontal='center')

        for column_number in range(1, sheet.max_column + 1):
            width = 15 if column_number == 1 else (12 if sheet.title == matrix_sheet else 24)
            sheet.column_dimensions[get_column_letter(column_number)].width = width
    workbook.save(path)

## Step 0: Load the starting constituent list

A change log only says what changed. It does not show every constituent that existed before the first recorded change. Therefore, one complete starting snapshot is required.

In [4]:
starting_constituents = pd.read_csv(STARTING_PATH, keep_default_na=False)
START_DATE = starting_constituents['as_of_date'].iloc[0]

print('Starting date:', START_DATE)
print('Starting tickers:', len(starting_constituents))
display(starting_constituents.head(10))

Starting date: 2015-03-20
Starting tickers: 107


,ticker,as_of_date,starting_point_note
0,AAL,2015-03-20,Last trading day before the first 2015 change;...
1,AAPL,2015-03-20,Last trading day before the first 2015 change;...
2,ADBE,2015-03-20,Last trading day before the first 2015 change;...
3,ADI,2015-03-20,Last trading day before the first 2015 change;...
4,ADP,2015-03-20,Last trading day before the first 2015 change;...
5,ADSK,2015-03-20,Last trading day before the first 2015 change;...
6,AKAM,2015-03-20,Last trading day before the first 2015 change;...
7,ALTR,2015-03-20,Last trading day before the first 2015 change;...
8,ALXN,2015-03-20,Last trading day before the first 2015 change;...
9,AMAT,2015-03-20,Last trading day before the first 2015 change;...


## Step 1: Read the official PDF sources

We first list the available PDFs. The GPT extraction step reads text from one PDF at a time.

In [5]:
pdf_paths = sorted(PDF_DIR.rglob('*.pdf'))
if MAX_PDFS is not None:
    pdf_paths = pdf_paths[:MAX_PDFS]

def read_pdf_text(path: Path) -> str:
    """Extract a bounded amount of text from one PDF."""
    reader = PdfReader(str(path))
    text = '\n'.join(page.extract_text() or '' for page in reader.pages)
    return text[:MAX_CHARS_PER_PDF]

pdf_inventory = pd.DataFrame({
    'source_file': [path.relative_to(PDF_DIR).as_posix() for path in pdf_paths],
    'file_size_bytes': [path.stat().st_size for path in pdf_paths],
})

print('PDFs selected:', len(pdf_inventory))
display(pdf_inventory.head(10))

PDFs selected: 72


,source_file,file_size_bytes
0,Corporate_actions/Broadcom_Avago_Transaction_A...,19778
1,Corporate_actions/Comcast_CMCSK_Reclassificati...,9861
2,Corporate_actions/Fiserv_FISV_to_FI_and_NYSE_T...,19136
3,Corporate_actions/Honeywell_Aerospace_Spin_Off...,235163
4,Corporate_actions/Honeywell_Solstice_Spin_Off_...,199567
5,Corporate_actions/Liberty_Global_LiLAC_Launch_...,43065
6,Corporate_actions/Liberty_Latin_America_Split_...,372896
7,Corporate_actions/Liberty_Latin_America_Split_...,55685
8,Corporate_actions/Liberty_Media_Tracking_Stock...,122831
9,Corporate_actions/Qurate_QVCA_to_QRTEA__2018-0...,130533


## Step 2: Create the constituent change table

Each output row represents one ticker-level action:

- `ADD`: the ticker enters the Nasdaq-100.
- `REMOVE`: the ticker leaves the Nasdaq-100.

The model must use the ticker that traded at the time and the first trading date when the change applied.

In [6]:
class ConstituentAction(BaseModel):
    effective_date: str
    action: Literal['ADD', 'REMOVE']
    ticker: str
    security_name: str
    change_reason: str
    related_ticker: str
    announcement_date: str
    effective_date_basis: str
    evidence_summary: str
    notes: str

class PDFResult(BaseModel):
    contains_constituent_changes: bool
    actions: list[ConstituentAction]

MODEL_INSTRUCTIONS = '''
Extract Nasdaq-100 constituent additions and removals from this official document.
Return only actions directly supported by the text. Do not guess.
Use the first trading date when the change applies, not just the announcement date.
Use the ticker that traded on that date.
Create one ADD or REMOVE row per security.
Use an empty string when optional text is unavailable.
Clearly label ticker changes, successor securities, and spin-offs.
'''

def ask_gpt_for_actions(client: OpenAI, pdf_path: Path) -> dict:
    """Extract structured constituent changes from one PDF."""
    source_file = pdf_path.relative_to(PDF_DIR).as_posix()
    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=MODEL_INSTRUCTIONS,
        input=f'SOURCE_FILE: {source_file}\n\n{read_pdf_text(pdf_path)}',
        text_format=PDFResult,
        reasoning={'effort': 'high'},
        max_output_tokens=12_000,
        store=False,
    )
    if response.output_parsed is None:
        raise ValueError(f'No structured result for {source_file}')
    return {'source_file': source_file, **response.output_parsed.model_dump()}

In [7]:
COLUMN_DEFINITIONS = pd.DataFrame([
    ('action_id', 'Unique ID for one ADD or REMOVE row.'),
    ('event_id', 'Groups ticker changes caused by the same index event.'),
    ('effective_date', 'First trading date when the index change applies.'),
    ('action', 'ADD enters the index; REMOVE leaves the index.'),
    ('ticker', 'Trading symbol used on the effective date.'),
    ('security_name', 'Company or security name associated with the ticker.'),
    ('change_reason', 'Reason such as annual reconstitution or ticker change.'),
    ('related_ticker', 'Ticker linked to the same add, removal, or ticker change.'),
    ('announcement_date', 'Date the change was publicly announced.'),
    ('source_references', 'IDs for the supporting PDFs or source records.'),
    ('effective_date_basis', 'Evidence used to choose the effective trading date.'),
    ('evidence_summary', 'Plain-English summary of the source evidence.'),
    ('verification_status', 'Whether primary sources were checked.'),
    ('notes', 'Identity, timing, or interpretation warnings.'),
], columns=['column_name', 'plain_english_definition'])

if RUN_GPT_EXTRACTION:
    api_key = os.getenv('OPENAI_API_KEY') or getpass.getpass('OPENAI_API_KEY: ')
    client = OpenAI(api_key=api_key)
    CACHE_DIR.mkdir(exist_ok=True)
    extracted_documents = []

    for number, pdf_path in enumerate(pdf_paths, start=1):
        source_file = pdf_path.relative_to(PDF_DIR).as_posix()
        cache_name = re.sub(r'[^A-Za-z0-9._-]+', '_', source_file) + '.json'
        cache_path = CACHE_DIR / cache_name

        if cache_path.exists():
            result = json.loads(cache_path.read_text(encoding='utf-8'))
        else:
            result = ask_gpt_for_actions(client, pdf_path)
            cache_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

        extracted_documents.append(result)
        print(f'[{number}/{len(pdf_paths)}] {source_file}')

    extracted_rows = []
    for document in extracted_documents:
        if not document['contains_constituent_changes']:
            continue
        for action in document['actions']:
            row = dict(action)
            row['ticker'] = row['ticker'].strip().upper()
            row['related_ticker'] = row['related_ticker'].strip().upper()
            row['source_references'] = document['source_file']
            extracted_rows.append(row)

    raw_changes = pd.DataFrame(extracted_rows)
    if raw_changes.empty:
        raise ValueError('No constituent changes were extracted from the PDFs.')

    # Keep one row per date, action, and ticker.
    raw_changes = raw_changes.drop_duplicates(['effective_date', 'action', 'ticker'])
    raw_changes['event_id'] = (
        raw_changes['effective_date'].str.replace('-', '', regex=False)
        + '_' + raw_changes['source_references'].map(lambda value: Path(value).stem[:35])
    )
    raw_changes['action_id'] = raw_changes['event_id'] + '_' + raw_changes['action'] + '_' + raw_changes['ticker']
    raw_changes['verification_status'] = 'gpt_extracted_needs_human_review'

    change_columns = [
        'action_id', 'event_id', 'effective_date', 'action', 'ticker',
        'security_name', 'change_reason', 'related_ticker',
        'announcement_date', 'source_references', 'effective_date_basis', 'evidence_summary',
        'verification_status', 'notes',
    ]
    changes = raw_changes[change_columns].copy()
    changes['_order'] = changes['action'].map({'REMOVE': 0, 'ADD': 1})
    changes = changes.sort_values(['effective_date', '_order', 'ticker']).drop(columns='_order')

    with pd.ExcelWriter(CHANGES_PATH, engine='openpyxl') as writer:
        changes.to_excel(writer, sheet_name='constituent_changes', index=False)
        COLUMN_DEFINITIONS.to_excel(writer, sheet_name='column_definitions', index=False)
    format_excel(CHANGES_PATH)
else:
    changes = pd.read_excel(CHANGES_PATH, sheet_name='constituent_changes', dtype=str).fillna('')

print('Constituent change rows:', len(changes))
display(changes.head(12))

Constituent change rows: 262


,action_id,event_id,effective_date,action,ticker,security_name,change_reason,related_ticker,announcement_date,source_references,effective_date_basis,evidence_summary,verification_status,notes
0,NDX-2015-03-23-WBA-EQIX-REMOVE-EQIX,NDX-2015-03-23-WBA-EQIX,2015-03-23,REMOVE,EQIX,Equinix,special_replacement,WBA,2015-03-13,NDX_2015_WBA,Official index effective date.,Nasdaq states WBA will replace EQIX prior to m...,primary_verified,
1,NDX-2015-03-23-WBA-EQIX-ADD-WBA,NDX-2015-03-23-WBA-EQIX,2015-03-23,ADD,WBA,Walgreens Boots Alliance,special_replacement,EQIX,2015-03-13,NDX_2015_WBA,Official index effective date.,Nasdaq states WBA will replace EQIX prior to m...,primary_verified,
2,NDX-2015-07-02-LILAC-ADD-LILA,NDX-2015-07-02-LILAC,2015-07-02,ADD,LILA,LiLAC / Liberty Latin America Class A,corporate_action_addition,,2015-07-01,ISSUER_2015_LILAC;NDX_REPORT_2015_EVAL_UNDATED,First regular trading date and Nasdaq retrospe...,LiLAC tracking shares were automatically added...,primary_verified,
3,NDX-2015-07-02-LILAC-ADD-LILAK,NDX-2015-07-02-LILAC,2015-07-02,ADD,LILAK,LiLAC / Liberty Latin America Class C,corporate_action_addition,,2015-07-01,ISSUER_2015_LILAC;NDX_REPORT_2015_EVAL_UNDATED,First regular trading date and Nasdaq retrospe...,LiLAC tracking shares were automatically added...,primary_verified,
4,NDX-2015-07-06-KRFT-KHC-REMOVE-KRFT,NDX-2015-07-06-KRFT-KHC,2015-07-06,REMOVE,KRFT,Kraft Foods Group,security_successor,KHC,2015-07-02,ISSUER_WEB_2015_KRAFT_HEINZ,First regular trading day after the merger com...,Kraft Foods Group ceased and the combined Kraf...,primary_verified,
5,NDX-2015-07-06-KRFT-KHC-ADD-KHC,NDX-2015-07-06-KRFT-KHC,2015-07-06,ADD,KHC,Kraft Heinz,security_successor,KRFT,2015-07-02,ISSUER_WEB_2015_KRAFT_HEINZ,First regular trading day after the merger com...,Kraft Foods Group ceased and the combined Kraf...,primary_verified,
6,NDX-2015-07-24-CTRX-REMOVE-CTRX,NDX-2015-07-24-CTRX,2015-07-24,REMOVE,CTRX,Catamaran,acquisition_deletion,,2015-07-23,NDX_2015_JD_CTRX,Nasdaq explicitly says CTRX is removed effecti...,The same announcement schedules JD for a later...,primary_verified,
7,NDX-2015-07-27-BMRN-DTV-REMOVE-DTV,NDX-2015-07-27-BMRN-DTV,2015-07-27,REMOVE,DTV,DIRECTV,special_replacement,BMRN,2015-07-22,NDX_2015_BMRN,Official index effective date.,Nasdaq states BMRN will replace DTV prior to m...,primary_verified,
8,NDX-2015-07-27-BMRN-DTV-ADD-BMRN,NDX-2015-07-27-BMRN-DTV,2015-07-27,ADD,BMRN,BioMarin Pharmaceutical,special_replacement,DTV,2015-07-22,NDX_2015_BMRN,Official index effective date.,Nasdaq states BMRN will replace DTV prior to m...,primary_verified,
9,NDX-2015-07-29-JD-ADD-JD,NDX-2015-07-29-JD,2015-07-29,ADD,JD,JD.com,delayed_replacement_addition,,2015-07-23,NDX_2015_JD_CTRX,Nasdaq explicitly schedules JD prior to market...,CTRX had already been removed effective July 24.,primary_verified,


## Step 3: Build the complete historical constituent matrix

This is the central calculation.

1. Start with the initial constituent tickers.
2. For each effective date, apply every REMOVE action.
3. Apply every ADD action.
4. Save `1` when a ticker is in the index and `0` when it is not.

In [8]:
active_tickers = set(starting_constituents['ticker'])
all_tickers = sorted(active_tickers | set(changes['ticker']))

matrix = pd.DataFrame({'ticker': all_tickers})
matrix[START_DATE] = matrix['ticker'].isin(active_tickers).astype(int)

for effective_date in sorted(changes['effective_date'].unique()):
    changes_on_date = changes.loc[changes['effective_date'].eq(effective_date)]

    # Apply removals before additions on the same date.
    for ticker in changes_on_date.loc[changes_on_date['action'].eq('REMOVE'), 'ticker']:
        if ticker not in active_tickers:
            raise ValueError(f'Cannot remove inactive ticker {ticker} on {effective_date}')
        active_tickers.remove(ticker)

    for ticker in changes_on_date.loc[changes_on_date['action'].eq('ADD'), 'ticker']:
        if ticker in active_tickers:
            raise ValueError(f'Cannot add active ticker {ticker} on {effective_date}')
        active_tickers.add(ticker)

    matrix[effective_date] = matrix['ticker'].isin(active_tickers).astype(int)

with pd.ExcelWriter(MATRIX_PATH, engine='openpyxl') as writer:
    matrix.to_excel(writer, sheet_name='historical_constituents_1_0', index=False)
format_excel(MATRIX_PATH, matrix_sheet='historical_constituents_1_0')

print('Historical ticker rows:', len(matrix))
print('Constituent snapshot dates:', matrix.shape[1] - 1)
display(matrix.head(10))

Historical ticker rows: 223
Constituent snapshot dates: 74


,ticker,2015-03-20,2015-03-23,2015-07-02,2015-07-06,2015-07-24,2015-07-27,2015-07-29,2015-08-03,2015-10-07,...,2025-07-28,2025-10-30,2025-11-10,2025-12-22,2026-01-20,2026-04-20,2026-05-18,2026-06-22,2026-06-29,2026-07-07
0,AAL,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,AAPL,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2,ABNB,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
3,ADBE,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4,ADI,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
5,ADP,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
6,ADSK,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
7,AEP,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
8,AKAM,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
9,ALAB,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1


## Step 4: Create the exit and ticker-change log

File 04 is a compact explanation table. It separates removals from the index from ticker or successor-security changes.

In [9]:
name_by_ticker = (
    changes.loc[changes['security_name'].ne(''), ['ticker', 'security_name']]
    .drop_duplicates('ticker', keep='last')
    .set_index('ticker')['security_name']
    .to_dict()
)

ticker_change_reasons = {'ticker_change', 'security_successor', 'share_class_reclassification'}
exit_change_rows = []

for removal in changes.loc[changes['action'].eq('REMOVE')].itertuples(index=False):
    additions = changes.loc[
        changes['event_id'].eq(removal.event_id) & changes['action'].eq('ADD'), 'ticker'
    ].tolist()
    is_ticker_change = removal.change_reason in ticker_change_reasons
    new_ticker = additions[0] if is_ticker_change and len(additions) == 1 else ''

    exit_change_rows.append({
        'effective_date': removal.effective_date,
        'event_id': removal.event_id,
        'change_type': 'TICKER_OR_SECURITY_CHANGE' if is_ticker_change else 'REMOVED_FROM_INDEX',
        'old_ticker': removal.ticker,
        'new_ticker': new_ticker,
        'replacement_ticker': '' if is_ticker_change else removal.related_ticker,
        'old_security_name': removal.security_name,
        'new_security_name': name_by_ticker.get(new_ticker, ''),
        'change_reason': removal.change_reason,
        'source_references': removal.source_references,
        'notes': removal.notes,
    })

exit_changes = pd.DataFrame(exit_change_rows).sort_values(['effective_date', 'change_type', 'old_ticker'])
exit_changes.to_csv(EXITS_PATH, index=False, encoding='utf-8-sig')

print('Index-exit or ticker-change rows:', len(exit_changes))
display(exit_changes.head(15))

Index-exit or ticker-change rows: 133


,effective_date,event_id,change_type,old_ticker,new_ticker,replacement_ticker,old_security_name,new_security_name,change_reason,source_references,notes
0,2015-03-23,NDX-2015-03-23-WBA-EQIX,REMOVED_FROM_INDEX,EQIX,,WBA,Equinix,,special_replacement,NDX_2015_WBA,
1,2015-07-06,NDX-2015-07-06-KRFT-KHC,TICKER_OR_SECURITY_CHANGE,KRFT,KHC,,Kraft Foods Group,Kraft Heinz,security_successor,ISSUER_WEB_2015_KRAFT_HEINZ,
2,2015-07-24,NDX-2015-07-24-CTRX,REMOVED_FROM_INDEX,CTRX,,,Catamaran,,acquisition_deletion,NDX_2015_JD_CTRX,
3,2015-07-27,NDX-2015-07-27-BMRN-DTV,REMOVED_FROM_INDEX,DTV,,BMRN,DIRECTV,,special_replacement,NDX_2015_BMRN,
4,2015-08-03,NDX-2015-08-03-SWKS-SIAL,REMOVED_FROM_INDEX,SIAL,,SWKS,Sigma-Aldrich,,special_replacement,NDX_2015_SWKS,
5,2015-10-07,NDX-2015-10-07-INCY-ALTR,REMOVED_FROM_INDEX,ALTR,,INCY,Altera,,special_replacement,NDX_2015_INCY,
6,2015-11-11,NDX-2015-11-11-PYPL-BRCM,REMOVED_FROM_INDEX,BRCM,,PYPL,Broadcom Corporation,,special_replacement,NDX_2015_PYPL,
7,2015-12-14,NDX-2015-12-14-CMCSK,TICKER_OR_SECURITY_CHANGE,CMCSK,,,Comcast Class A Special,,share_class_reclassification,ISSUER_2015_CMCSK,
8,2015-12-21,NDX-2015-12-21-ANNUAL,REMOVED_FROM_INDEX,CHRW,,,C.H. Robinson Worldwide,,annual_reconstitution,NDX_2015_ANNUAL,
9,2015-12-21,NDX-2015-12-21-ANNUAL,REMOVED_FROM_INDEX,EXPD,,,Expeditors International,,annual_reconstitution,NDX_2015_ANNUAL,


## Step 5: Create the current-constituents-only workbook

We select tickers whose value is `1` in the final matrix column. This produces a convenient view of the latest verified constituents.

> This step intentionally removes historical non-survivors. It is useful for inspection, but it is **not** the correct historical backtest universe.

In [10]:
final_date = matrix.columns[-1]
current_tickers = matrix.loc[matrix[final_date].eq(1), 'ticker'].sort_values().tolist()

current_only_matrix = (
    matrix.loc[matrix['ticker'].isin(current_tickers)]
    .sort_values('ticker')
    .reset_index(drop=True)
)

current_constituents = pd.DataFrame({
    'ticker': current_tickers,
    'security_name': [name_by_ticker.get(ticker, '') for ticker in current_tickers],
    'as_of_date': final_date,
})

starting_ticker_set = set(starting_constituents['ticker'])
current_ticker_set = set(current_tickers)
shared_tickers = starting_ticker_set & current_ticker_set
start_only_tickers = starting_ticker_set - current_ticker_set
current_only_tickers = current_ticker_set - starting_ticker_set

constituent_comparison = pd.DataFrame([
    ('Same ticker in both lists', len(shared_tickers), len(shared_tickers) / len(starting_ticker_set) * 100, '2015 starting list'),
    ('2015 ticker absent from latest list', len(start_only_tickers), len(start_only_tickers) / len(starting_ticker_set) * 100, '2015 starting list'),
    ('Latest ticker absent from 2015 list', len(current_only_tickers), len(current_only_tickers) / len(current_ticker_set) * 100, 'latest constituent list'),
], columns=['comparison_group', 'ticker_count', 'percentage', 'percentage_denominator'])
constituent_comparison['percentage'] = constituent_comparison['percentage'].round(1)

with pd.ExcelWriter(CURRENT_PATH, engine='openpyxl') as writer:
    current_only_matrix.to_excel(writer, sheet_name='current_only_matrix_1_0', index=False)
    current_constituents.to_excel(writer, sheet_name='current_constituents', index=False)
format_excel(CURRENT_PATH, matrix_sheet='current_only_matrix_1_0')

print('Latest verified change date:', final_date)
print('Current constituent count:', len(current_tickers))
display(constituent_comparison)
display(current_constituents.head(20))

Latest verified change date:

 2026-07-07
Current constituent count: 103


,comparison_group,ticker_count,percentage,percentage_denominator
0,Same ticker in both lists,43,40.2,2015 starting list
1,2015 ticker absent from latest list,64,59.8,2015 starting list
2,Latest ticker absent from 2015 list,60,58.3,latest constituent list


,ticker,security_name,as_of_date
0,AAPL,,2026-07-07
1,ABNB,Airbnb,2026-07-07
2,ADBE,,2026-07-07
3,ADI,,2026-07-07
4,ADP,,2026-07-07
5,ADSK,,2026-07-07
6,AEP,American Electric Power,2026-07-07
7,ALAB,Astera Labs,2026-07-07
8,ALNY,Alnylam Pharmaceuticals,2026-07-07
9,AMAT,,2026-07-07


## Validation

In [11]:
checks = pd.DataFrame([
    ('Starting ticker count', len(starting_constituents), EXPECTED_STARTING_TICKERS),
    ('Constituent change rows', len(changes), EXPECTED_CHANGE_ROWS),
    ('Historical ticker rows', len(matrix), EXPECTED_HISTORICAL_TICKERS),
    ('Constituent snapshot dates', matrix.shape[1] - 1, EXPECTED_DATE_COLUMNS),
    ('Current constituent count', len(current_tickers), EXPECTED_CURRENT_CONSTITUENTS),
    ('Ticker symbols in both lists', len(shared_tickers), EXPECTED_SHARED_TICKERS),
    ('2015-only ticker symbols', len(start_only_tickers), EXPECTED_START_ONLY_TICKERS),
    ('Latest-only ticker symbols', len(current_only_tickers), EXPECTED_CURRENT_ONLY_TICKERS),
], columns=['check', 'actual', 'expected'])
checks['passed'] = checks['actual'].eq(checks['expected'])
display(checks)

assert matrix.drop(columns='ticker').isin([0, 1]).all().all(), 'Matrix contains a value other than 0 or 1.'
assert not changes['action_id'].duplicated().any(), 'Duplicate action_id found.'
assert checks['passed'].all(), 'The result does not match the validated reference. Review the source PDFs and change rows.'

print('PASS: all core outputs match the validated reference.')

,check,actual,expected,passed
0,Starting ticker count,107,107,True
1,Constituent change rows,262,262,True
2,Historical ticker rows,223,223,True
3,Constituent snapshot dates,74,74,True
4,Current constituent count,103,103,True
5,Ticker symbols in both lists,43,43,True
6,2015-only ticker symbols,64,64,True
7,Latest-only ticker symbols,60,60,True


PASS: all core outputs match the validated reference.


## How to Use the Results

- **File 02** lists every verified constituent addition and removal.
- **File 03** is the authoritative historical universe for survivorship-bias-aware backtests.
- **File 04** explains index removals, replacements, and ticker changes.
- **File 05** contains only the latest constituents. Do not use it as a historical universe.

If GPT extraction fails validation, review the cited source, correct the change row, and rebuild the matrix.